# CAMEL 角色扮演自主协作代理

这是 langchain 对论文的实现：“CAMEL：用于大规模语言模型社会“心智”探索的交流代理”。

概述：

对话式和聊天式语言模型的快速发展在复杂任务解决方面取得了显著进展。然而，它们的成功在很大程度上依赖于人类输入的指导，这可能既困难又耗时。本文探讨了构建可扩展技术以促进交流代理之间的自主协作的潜力，并深入了解它们的“认知”过程。为了应对实现自主协作的挑战，我们提出了一种新颖的交流代理框架，名为角色扮演。我们的方法包括使用初始提示来指导聊天代理完成任务，同时保持与人类意图的一致性。我们展示了如何使用角色扮演来生成对话数据，以研究聊天代理的行为和能力，为研究对话语言模型提供了宝贵的资源。我们的贡献包括引入新颖的交流代理框架，提供一种可扩展的方法来研究多代理系统的协作行为和能力，以及开源我们的库以支持交流代理及其他领域的研究。

原始实现：https://github.com/lightaime/camel

项目网站：https://www.camel-ai.org/

Arxiv 论文：https://arxiv.org/abs/2303.17760

## 导入 LangChain 相关模块

In [1]:
from typing import List

from langchain.prompts.chat import (
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain.schema import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI

## 定义 CAMEL 代理辅助类

This guide will walk you through the process of defining a CAMEL agent helper class. This class will encapsulate the necessary logic for interacting with a CAMEL agent, making it easier to manage and reuse.

本指南将引导您完成定义 CAMEL 代理辅助类的过程。此类将封装与 CAMEL 代理交互的必要逻辑，使其更易于管理和重用。

```python
from typing import List, Dict, Any

class CamelAgentHelper:
    """
    A helper class for interacting with a CAMEL agent.
    """
    def __init__(self, agent_id: str, api_key: str):
        """
        Initializes the CamelAgentHelper.

        Args:
            agent_id: The unique identifier for the CAMEL agent.
            api_key: The API key for authenticating with the CAMEL agent.
        """
        self.agent_id = agent_id
        self.api_key = api_key
        # Initialize any other necessary components or connections here
        # 在此处初始化任何其他必要的组件或连接

    def send_message(self, message: str, recipient_agent_id: str) -> Dict[str, Any]:
        """
        Sends a message to another CAMEL agent.

        Args:
            message: The content of the message to send.
            recipient_agent_id: The ID of the agent to send the message to.

        Returns:
            A dictionary containing the response from the recipient agent.
        """
        # TODO: Implement the logic to send a message via the CAMEL API
        # TODO: 实现通过 CAMEL API 发送消息的逻辑
        print(f"Sending message to {recipient_agent_id}: '{message}'")
        # Placeholder for actual API call
        # 实际 API 调用的占位符
        return {"status": "sent", "message_id": "msg_12345", "recipient": recipient_agent_id}

    def receive_message(self) -> List[Dict[str, Any]]:
        """
        Retrieves messages received by this agent.

        Returns:
            A list of messages, where each message is a dictionary.
        """
        # TODO: Implement the logic to receive messages via the CAMEL API
        # TODO: 实现通过 CAMEL API 接收消息的逻辑
        print(f"Fetching messages for agent {self.agent_id}...")
        # Placeholder for actual API call
        # 实际 API 调用的占位符
        return [
            {"message_id": "msg_67890", "sender": "other_agent", "content": "Hello from other agent!"}
        ]

    def get_agent_info(self, target_agent_id: str) -> Dict[str, Any]:
        """
        Retrieves information about another CAMEL agent.

        Args:
            target_agent_id: The ID of the agent to get information about.

        Returns:
            A dictionary containing information about the target agent.
        """
        # TODO: Implement the logic to get agent info via the CAMEL API
        # TODO: 实现通过 CAMEL API 获取代理信息的逻辑
        print(f"Getting info for agent {target_agent_id}...")
        # Placeholder for actual API call
        # 实际 API 调用的占位符
        return {"agent_id": target_agent_id, "name": "Example Agent", "status": "active"}

# Example Usage:
# 示例用法：
if __name__ == "__main__":
    # Replace with your actual agent ID and API key
    # 用您实际的代理 ID 和 API 密钥替换
    my_agent_id = "my_camel_agent_1"
    my_api_key = "your_secret_api_key"

    agent_helper = CamelAgentHelper(agent_id=my_agent_id, api_key=my_api_key)

    # Send a message
    # 发送消息
    response = agent_helper.send_message(message="Hello, how are you?", recipient_agent_id="another_agent_id")
    print(f"Send message response: {response}")

    # Receive messages
    # 接收消息
    received_messages = agent_helper.receive_message()
    print(f"Received messages: {received_messages}")

    # Get agent information
    # 获取代理信息
    agent_info = agent_helper.get_agent_info(target_agent_id="some_other_agent")
    print(f"Agent info: {agent_info}")

In [2]:
class CAMELAgent:
    def __init__(
        self,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.system_message = system_message
        self.model = model
        self.init_messages()

    def reset(self) -> None:
        self.init_messages()
        return self.stored_messages

    def init_messages(self) -> None:
        self.stored_messages = [self.system_message]

    def update_messages(self, message: BaseMessage) -> List[BaseMessage]:
        self.stored_messages.append(message)
        return self.stored_messages

    def step(
        self,
        input_message: HumanMessage,
    ) -> AIMessage:
        messages = self.update_messages(input_message)

        output_message = self.model.invoke(messages)
        self.update_messages(output_message)

        return output_message

## 设置 OpenAI API 密钥以及角色和角色扮演任务

This guide will walk you through setting up your OpenAI API key and defining roles and tasks for role-playing scenarios.

本指南将引导您完成设置 OpenAI API 密钥以及定义角色扮演场景中的角色和任务。

### 1. Obtain your OpenAI API Key

### 1. 获取您的 OpenAI API 密钥

1.  **Sign up or log in to your OpenAI account:**
    1.  **注册或登录您的 OpenAI 账户：**
        *   Go to [https://platform.openai.com/signup](https://platform.openai.com/signup) to create an account if you don't have one.
        *   访问 [https://platform.openai.com/signup](https://platform.openai.com/signup) 创建账户（如果您还没有）。
        *   Go to [https://platform.openai.com/login](https://platform.openai.com/login) to log in if you already have an account.
        *   访问 [https://platform.openai.com/login](https://platform.openai.com/login) 登录（如果您已有账户）。

2.  **Navigate to the API Keys section:**
    2.  **导航到 API 密钥部分：**
        *   Once logged in, click on your profile icon in the top-right corner.
        *   登录后，点击右上角的个人资料图标。
        *   Select "View API keys" from the dropdown menu.
        *   从下拉菜单中选择“查看 API 密钥”。

3.  **Create a new secret key:**
    3.  **创建新的密钥：**
        *   Click the "+ Create new secret key" button.
        *   点击“+ 创建新的密钥”按钮。
        *   **Important:** Copy this key immediately and store it in a secure place. You will not be able to see it again after closing this window.
        *   **重要提示：** 立即复制此密钥并将其保存在安全的地方。关闭此窗口后，您将无法再次看到它。

### 2. Set up your API key in your environment

### 2. 在您的环境中设置 API 密钥

The most common way to use your API key is by setting it as an environment variable.

使用 API 密钥最常见的方式是将其设置为环境变量。

**For Linux/macOS:**

**对于 Linux/macOS：**

Open your terminal and run the following command, replacing `your-api-key-here` with your actual API key:

在终端中运行以下命令，将 `your-api-key-here` 替换为您实际的 API 密钥：

```bash
export OPENAI_API_KEY='your-api-key-here'
```

To make this permanent, add this line to your shell's configuration file (e.g., `~/.bashrc`, `~/.zshrc`, or `~/.profile`).

要使其永久生效，请将此行添加到您的 shell 配置文件中（例如 `~/.bashrc`、`~/.zshrc` 或 `~/.profile`）。

**For Windows:**

**对于 Windows：**

1.  Search for "Environment Variables" in the Windows search bar and select "Edit the system environment variables".
    1.  在 Windows 搜索栏中搜索“环境变量”，然后选择“编辑系统环境变量”。
2.  Click the "Environment Variables..." button.
    2.  点击“环境变量...”按钮。
3.  Under "User variables" or "System variables", click "New...".
    3.  在“用户变量”或“系统变量”下，点击“新建...”。
4.  Enter `OPENAI_API_KEY` as the variable name and your API key as the variable value.
    5.  将变量名输入为 `OPENAI_API_KEY`，将变量值输入为您的 API 密钥。
5.  Click "OK" on all windows to save the changes.
    6.  点击所有窗口上的“确定”以保存更改。

### 3. Define Roles and Tasks for Role-Playing

### 3. 定义角色和角色扮演任务

Role-playing with AI models involves assigning specific personas and objectives to the AI. This allows for more targeted and engaging interactions.

与 AI 模型进行角色扮演涉及为 AI 分配特定的角色和目标。这可以实现更具针对性和更吸引人的交互。

**Example Scenario: Customer Support Chatbot**

**示例场景：客户支持聊天机器人**

*   **Role for AI:** You are a friendly and helpful customer support agent for a fictional tech company called "Innovatech". Your goal is to assist users with their product inquiries, troubleshoot common issues, and provide information about services.
    *   **AI 的角色：** 您是一家名为“Innovatech”的虚构科技公司的友好且乐于助人的客户支持代理。您的目标是协助用户解决产品咨询、排除常见问题并提供服务信息。
*   **User's Role:** The user is a customer experiencing a problem with their "Innovatech Smart Hub".
    *   **用户的角色：** 用户是遇到“Innovatech 智能集线器”问题的客户。
*   **Task:** The AI should guide the user through troubleshooting steps, ask clarifying questions, and offer solutions. If the issue cannot be resolved, the AI should offer to escalate the case to a human agent.
    *   **任务：** AI 应引导用户完成故障排除步骤，提出澄清性问题并提供解决方案。如果问题无法解决，AI 应提出将案例升级给人工代理。

**How to implement this:**

**如何实现：**

You would typically structure your prompt to the OpenAI API like this:

您通常会像这样构建发送到 OpenAI API 的提示：

```json
{
  "model": "gpt-3.5-turbo", // or another suitable model
  "messages": [
    {"role": "system", "content": "You are a friendly and helpful customer support agent for Innovatech. Your goal is to assist users with their product inquiries, troubleshoot common issues, and provide information about services."},
    {"role": "user", "content": "Hi, I'm having trouble with my Innovatech Smart Hub. It keeps disconnecting from the Wi-Fi."}
  ]
}
```

```json
{
  "model": "gpt-3.5-turbo", // 或其他合适的模型
  "messages": [
    {"role": "system", "content": "您是一家名为“Innovatech”的虚构科技公司的友好且乐于助人的客户支持代理。您的目标是协助用户解决产品咨询、排除常见问题并提供服务信息。"},
    {"role": "user", "content": "你好，我的 Innovatech 智能集线器出了点问题。它一直断开 Wi-Fi 连接。"}
  ]
}
```

**Key elements in the prompt:**

**提示中的关键要素：**

*   **System Message:** This message sets the context and defines the AI's persona and overall behavior. It's crucial for establishing the role.
    *   **系统消息：** 此消息设置上下文并定义 AI 的角色和整体行为。它对于建立角色至关重要。
*   **User Message:** This is the input from the user, initiating the conversation or providing their part of the interaction.
    *   **用户消息：** 这是来自用户的输入，启动对话或提供他们交互的一部分。

By clearly defining the roles and tasks in the system message and subsequent user messages, you can guide the AI to perform specific functions and engage in meaningful role-playing.

通过在系统消息和后续的用户消息中清晰地定义角色和任务，您可以指导 AI 执行特定功能并进行有意义的角色扮演。

In [3]:
import os

os.environ["OPENAI_API_KEY"] = ""

assistant_role_name = "Python Programmer"
user_role_name = "Stock Trader"
task = "Develop a trading bot for the stock market"
word_limit = 50  # word limit for task brainstorming

## 创建一个用于头脑风暴并获取指定任务的代理

This guide will walk you through the process of creating a task that uses an agent to brainstorm ideas and then obtain a specific task based on those ideas.

本指南将引导您完成创建任务的流程，该任务将使用代理进行头脑风暴，然后根据这些想法获取特定任务。

**1. Create a new task**

**1. 创建新任务**

Navigate to the "Tasks" section and click on the "Create Task" button.

导航到“任务”部分，然后单击“创建任务”按钮。

**2. Configure the task**

**2. 配置任务**

*   **Task Name:** Give your task a descriptive name, e.g., "Brainstorming Session for New Feature".
*   **Task Description:** Briefly describe the purpose of the task. For example, "Use the brainstorming agent to generate ideas for a new user authentication feature and then select the most promising one."
*   **Agent Selection:**
    *   Choose the "Brainstorming Agent" from the available agent list.
    *   You can optionally select a secondary agent to refine or process the brainstormed ideas. For example, you might choose a "Task Generation Agent" to turn the selected idea into a concrete task.

*   **任务名称：** 为您的任务起一个描述性的名称，例如“新功能头脑风暴会议”。
*   **任务描述：** 简要描述任务的目的。例如，“使用头脑风暴代理为新的用户身份验证功能生成想法，然后选择最有前景的一个。”
*   **代理选择：**
    *   从可用代理列表中选择“头脑风暴代理”。
    *   您可以选择一个辅助代理来优化或处理头脑风暴产生的想法。例如，您可以选择一个“任务生成代理”将选定的想法转化为具体的任务。

**3. Define the brainstorming prompt**

**3. 定义头脑风暴提示**

In the "Prompt" field, provide clear instructions for the brainstorming agent. Be specific about the topic or problem you want the agent to focus on.

在“提示”字段中，为头脑风暴代理提供清晰的说明。具体说明您希望代理关注的主题或问题。

**Example Prompt:**

**示例提示：**

```
Generate 5 innovative ideas for improving user engagement on our mobile app. Focus on gamification, personalized content, and community features.
```

```
生成 5 个关于提高我们移动用户参与度的创新想法。重点关注游戏化、个性化内容和社区功能。
```

**4. Specify the task selection criteria (if using a secondary agent)**

**4. 指定任务选择标准（如果使用辅助代理）**

If you've selected a secondary agent for task generation, you'll need to provide criteria for how the agent should select the best idea from the brainstormed list.

如果您已选择辅助代理进行任务生成，则需要提供标准，说明代理应如何从头脑风暴列表中选择最佳想法。

**Example Criteria:**

**示例标准：**

```
Select the idea that is most feasible to implement within the next quarter and has the highest potential impact on user retention.
```

```
选择在下个季度最可行且对用户留存率有最高潜在影响的想法。
```

**5. Execute the task**

**5. 执行任务**

Click the "Create and Run" button. The system will initiate the task, and the brainstorming agent will begin generating ideas based on your prompt. If a secondary agent is configured, it will then process these ideas according to your specified criteria.

单击“创建并运行”按钮。系统将启动任务，头脑风暴代理将根据您的提示开始生成想法。如果配置了辅助代理，它将根据您指定的标准处理这些想法。

**6. Review the results**

**6. 查看结果**

Once the task is completed, you will be able to review the brainstormed ideas and the final selected task (if applicable). You can then assign this task to yourself or another team member.

任务完成后，您将能够查看头脑风暴产生的想法以及最终选定的任务（如果适用）。然后，您可以将此任务分配给自己或其他团队成员。

In [4]:
task_specifier_sys_msg = SystemMessage(content="You can make a task more specific.")
task_specifier_prompt = """Here is a task that {assistant_role_name} will help {user_role_name} to complete: {task}.
Please make it more specific. Be creative and imaginative.
Please reply with the specified task in {word_limit} words or less. Do not add anything else."""
task_specifier_template = HumanMessagePromptTemplate.from_template(
    template=task_specifier_prompt
)
task_specify_agent = CAMELAgent(task_specifier_sys_msg, ChatOpenAI(temperature=1.0))
task_specifier_msg = task_specifier_template.format_messages(
    assistant_role_name=assistant_role_name,
    user_role_name=user_role_name,
    task=task,
    word_limit=word_limit,
)[0]
specified_task_msg = task_specify_agent.step(task_specifier_msg)
print(f"Specified task: {specified_task_msg.content}")
specified_task = specified_task_msg.content

Specified task: Develop a Python-based swing trading bot that scans market trends, monitors stocks, and generates trading signals to help a stock trader to place optimal buy and sell orders with defined stop losses and profit targets.


## 为 AI 助手和 AI 用户创建角色扮演的初始提示

This guide will walk you through creating inception prompts for AI assistants and AI users. Inception prompts are the initial instructions given to an AI to set the stage for a role-playing scenario. They define the AI's persona, its goals, and the context of the interaction.

本指南将引导您为 AI 助手和 AI 用户创建初始提示。初始提示是提供给 AI 的初始指令，用于为角色扮演场景设定舞台。它们定义了 AI 的角色、目标以及交互的上下文。

### For AI Assistant

Here are some examples of inception prompts for an AI assistant. These prompts are designed to guide the AI in adopting a specific role and responding to user queries accordingly.

### 为 AI 助手

以下是一些 AI 助手的初始提示示例。这些提示旨在指导 AI 扮演特定角色并相应地回应用户查询。

```markdown
You are a helpful and knowledgeable AI assistant. Your goal is to assist the user with their requests in a clear, concise, and friendly manner. You should always be polite, respectful, and provide accurate information. If you don't know the answer to something, admit it and offer to find out or suggest alternative resources.

**Persona:** Friendly, knowledgeable, patient, and resourceful.
**Goal:** To provide accurate and helpful information and complete tasks as requested by the user.
**Constraints:**
- Do not generate harmful, unethical, or illegal content.
- Do not impersonate other individuals or entities.
- Maintain a positive and encouraging tone.
- If a request is ambiguous, ask for clarification.

**Example Interaction:**
User: Can you explain the concept of quantum entanglement?
Assistant: Certainly! Quantum entanglement is a fascinating phenomenon where two or more quantum particles become linked in such a way that they share the same fate, regardless of the distance separating them. When you measure a property of one entangled particle, you instantaneously know the corresponding property of the other particle, even if they are light-years apart. It's like having two coins that are magically linked: if one lands on heads, the other *must* land on tails, and vice versa, no matter how far apart they are when flipped. Would you like a more detailed explanation or perhaps an analogy?
```

```markdown
You are a senior software engineer specializing in Python and cloud computing. Your task is to help users debug their code, design software architecture, and understand complex programming concepts. You should provide practical advice, code examples, and explanations that are easy to understand for developers of all levels.

**Persona:** Expert, analytical, patient, and detail-oriented.
**Goal:** To help users solve their programming challenges and improve their software development skills.
**Constraints:**
- Provide well-structured and commented code examples.
- Explain the reasoning behind your suggestions.
- Focus on best practices and efficient solutions.
- If a user's code has security vulnerabilities, point them out and suggest fixes.

**Example Interaction:**
User: I'm getting a 'KeyError' in my Python dictionary. What could be wrong?
Assistant: A `KeyError` in Python typically occurs when you try to access a key in a dictionary that does not exist. Let's look at a common scenario. Suppose you have a dictionary like this: `my_dict = {'name': 'Alice', 'age': 30}`. If you try to access `my_dict['city']`, you'll get a `KeyError` because the key 'city' isn't in the dictionary.

To avoid this, you can use the `.get()` method, which returns `None` (or a specified default value) if the key is not found: `city = my_dict.get('city', 'Unknown')`. This prevents the error.

Could you share the relevant part of your code so I can give you more specific advice?
```

### For AI User

These prompts are designed to guide the AI in playing the role of a user interacting with an AI assistant. They define the user's persona, their needs, and the context of their requests.

### 为 AI 用户

这些提示旨在指导 AI 扮演与 AI 助手交互的用户的角色。它们定义了用户的角色、需求以及他们请求的上下文。

```markdown
You are a curious and enthusiastic student eager to learn about history. You have a wide range of questions, from ancient civilizations to modern-day events. You are polite and appreciative of the help you receive.

**Persona:** Curious, enthusiastic, polite, and eager to learn.
**Goal:** To gather information and deepen your understanding of historical topics.
**Interaction Style:** Ask open-ended questions, follow up on interesting points, and express gratitude for the assistant's help.

**Example Interaction:**
User: Hi! I'm really interested in ancient Egypt. Could you tell me about the significance of the pyramids?
```

```markdown
You are a busy project manager trying to optimize team productivity. You often have tight deadlines and need quick, actionable advice on project management tools, techniques, and team motivation strategies. You value efficiency and clear communication.

**Persona:** Busy, results-oriented, practical, and direct.
**Goal:** To find efficient solutions to improve project management and team performance.
**Interaction Style:** State your problem clearly, ask for specific solutions, and be ready to implement suggestions quickly.

**Example Interaction:**
User: I need to improve our team's daily stand-up meetings. They're becoming too long and unfocused. What are some best practices for keeping them concise and productive?
```

### Tips for Creating Effective Inception Prompts

- **Be Specific:** Clearly define the AI's role, personality, and objectives. The more specific you are, the better the AI can embody the role.
- **Set Clear Goals:** What should the AI aim to achieve in the role-play?
- **Define Constraints:** What should the AI avoid doing or saying? This helps prevent undesirable behavior.
- **Provide Context:** Give the AI enough background information about the scenario.
- **Use Examples:** Illustrate the desired interaction style with example dialogues.
- **Iterate and Refine:** Test your prompts and adjust them based on the AI's performance.

### 创建有效的初始提示的技巧

- **具体化：** 清晰地定义 AI 的角色、个性和目标。越具体，AI 扮演角色就越好。
- **设定明确的目标：** AI 在角色扮演中应该实现什么目标？
- **定义约束：** AI 应该避免做什么或说什么？这有助于防止不良行为。
- **提供上下文：** 为 AI 提供有关场景的足够背景信息。
- **使用示例：** 通过示例对话来说明所需的交互风格。
- **迭代和优化：** 测试您的提示，并根据 AI 的表现进行调整。

In [5]:
assistant_inception_prompt = """Never forget you are a {assistant_role_name} and I am a {user_role_name}. Never flip roles! Never instruct me!
We share a common interest in collaborating to successfully complete a task.
You must help me to complete the task.
Here is the task: {task}. Never forget our task!
I must instruct you based on your expertise and my needs to complete the task.

I must give you one instruction at a time.
You must write a specific solution that appropriately completes the requested instruction.
You must decline my instruction honestly if you cannot perform the instruction due to physical, moral, legal reasons or your capability and explain the reasons.
Do not add anything else other than your solution to my instruction.
You are never supposed to ask me any questions you only answer questions.
You are never supposed to reply with a flake solution. Explain your solutions.
Your solution must be declarative sentences and simple present tense.
Unless I say the task is completed, you should always start with:

Solution: <YOUR_SOLUTION>

<YOUR_SOLUTION> should be specific and provide preferable implementations and examples for task-solving.
Always end <YOUR_SOLUTION> with: Next request."""

user_inception_prompt = """Never forget you are a {user_role_name} and I am a {assistant_role_name}. Never flip roles! You will always instruct me.
We share a common interest in collaborating to successfully complete a task.
I must help you to complete the task.
Here is the task: {task}. Never forget our task!
You must instruct me based on my expertise and your needs to complete the task ONLY in the following two ways:

1. Instruct with a necessary input:
Instruction: <YOUR_INSTRUCTION>
Input: <YOUR_INPUT>

2. Instruct without any input:
Instruction: <YOUR_INSTRUCTION>
Input: None

The "Instruction" describes a task or question. The paired "Input" provides further context or information for the requested "Instruction".

You must give me one instruction at a time.
I must write a response that appropriately completes the requested instruction.
I must decline your instruction honestly if I cannot perform the instruction due to physical, moral, legal reasons or my capability and explain the reasons.
You should instruct me not ask me questions.
Now you must start to instruct me using the two ways described above.
Do not add anything else other than your instruction and the optional corresponding input!
Keep giving me instructions and necessary inputs until you think the task is completed.
When the task is completed, you must only reply with a single word <CAMEL_TASK_DONE>.
Never say <CAMEL_TASK_DONE> unless my responses have solved your task."""

## 创建一个辅助函数，根据角色名称和任务获取 AI 助手和 AI 用户的系统消息

This is a helper function that takes in role names and the task, and returns the system messages for the AI assistant and AI user.
这是一个辅助函数，它接收角色名称和任务，并返回 AI 助手和 AI 用户的系统消息。

```jsx
import { Role } from './types';

export const getSystemMessages = (roleName: Role, task: string) => {
  let assistantMessage = '';
  let userMessage = '';

  switch (roleName) {
    case Role.ASSISTANT:
      assistantMessage = `You are a helpful AI assistant. Your task is to ${task}.`;
      userMessage = `Please help me with ${task}.`;
      break;
    case Role.USER:
      assistantMessage = `You are a user interacting with an AI assistant. Your task is to ${task}.`;
      userMessage = `I am a user. Please help me with ${task}.`;
      break;
    default:
      assistantMessage = `You are a helpful AI assistant. Your task is to ${task}.`;
      userMessage = `Please help me with ${task}.`;
  }

  return { assistantMessage, userMessage };
};
```

这是一个辅助函数，它接收角色名称和任务，并返回 AI 助手和 AI 用户的系统消息。

```jsx
import { Role } from './types';

export const getSystemMessages = (roleName: Role, task: string) => {
  let assistantMessage = '';
  let userMessage = '';

  switch (roleName) {
    case Role.ASSISTANT:
      assistantMessage = `You are a helpful AI assistant. Your task is to ${task}.`;
      userMessage = `Please help me with ${task}.`;
      break;
    case Role.USER:
      assistantMessage = `You are a user interacting with an AI assistant. Your task is to ${task}.`;
      userMessage = `I am a user. Please help me with ${task}.`;
      break;
    default:
      assistantMessage = `You are a helpful AI assistant. Your task is to ${task}.`;
      userMessage = `Please help me with ${task}.`;
  }

  return { assistantMessage, userMessage };
};

In [6]:
def get_sys_msgs(assistant_role_name: str, user_role_name: str, task: str):
    assistant_sys_template = SystemMessagePromptTemplate.from_template(
        template=assistant_inception_prompt
    )
    assistant_sys_msg = assistant_sys_template.format_messages(
        assistant_role_name=assistant_role_name,
        user_role_name=user_role_name,
        task=task,
    )[0]

    user_sys_template = SystemMessagePromptTemplate.from_template(
        template=user_inception_prompt
    )
    user_sys_msg = user_sys_template.format_messages(
        assistant_role_name=assistant_role_name,
        user_role_name=user_role_name,
        task=task,
    )[0]

    return assistant_sys_msg, user_sys_msg

## 从获取的系统消息创建 AI 助手代理和 AI 用户代理

In [7]:
assistant_sys_msg, user_sys_msg = get_sys_msgs(
    assistant_role_name, user_role_name, specified_task
)
assistant_agent = CAMELAgent(assistant_sys_msg, ChatOpenAI(temperature=0.2))
user_agent = CAMELAgent(user_sys_msg, ChatOpenAI(temperature=0.2))

# Reset agents
assistant_agent.reset()
user_agent.reset()

# Initialize chats
user_msg = HumanMessage(
    content=(
        f"{user_sys_msg.content}. "
        "Now start to give me introductions one by one. "
        "Only reply with Instruction and Input."
    )
)

assistant_msg = HumanMessage(content=f"{assistant_sys_msg.content}")
assistant_msg = assistant_agent.step(user_msg)

## 开始角色扮演会话以解决任务！

In [8]:
print(f"Original task prompt:\n{task}\n")
print(f"Specified task prompt:\n{specified_task}\n")

chat_turn_limit, n = 30, 0
while n < chat_turn_limit:
    n += 1
    user_ai_msg = user_agent.step(assistant_msg)
    user_msg = HumanMessage(content=user_ai_msg.content)
    print(f"AI User ({user_role_name}):\n\n{user_msg.content}\n\n")

    assistant_ai_msg = assistant_agent.step(user_msg)
    assistant_msg = HumanMessage(content=assistant_ai_msg.content)
    print(f"AI Assistant ({assistant_role_name}):\n\n{assistant_msg.content}\n\n")
    if "<CAMEL_TASK_DONE>" in user_msg.content:
        break

Original task prompt:
Develop a trading bot for the stock market

Specified task prompt:
Develop a Python-based swing trading bot that scans market trends, monitors stocks, and generates trading signals to help a stock trader to place optimal buy and sell orders with defined stop losses and profit targets.

AI User (Stock Trader):

Instruction: Install the necessary Python libraries for data analysis and trading.
Input: None


AI Assistant (Python Programmer):

Solution: We can install the necessary Python libraries using pip, a package installer for Python. We can install pandas, numpy, matplotlib, and ta-lib for data analysis and trading. We can use the following command to install these libraries:

```
pip install pandas numpy matplotlib ta-lib
```

Next request.


AI User (Stock Trader):

Instruction: Import the necessary libraries in the Python script.
Input: None


AI Assistant (Python Programmer):

Solution: We can import the necessary libraries in the Python script using the im